In [51]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dense, Activation, Input
from tensorflow.keras.models import Model
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error

In [ ]:
student_performance = fetch_ucirepo(id=320)
# carrego os dados do dataset
x = student_performance.data.features
y = student_performance.data.targets
# objetivo é prever 3 saidas, 1 nota para 3 periodos difrentes

In [53]:
x

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,yes,no,no,4,3,4,1,1,3,4
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,yes,yes,no,5,3,3,1,1,3,2
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,yes,yes,no,4,3,2,2,3,3,6
3,GP,F,15,U,GT3,T,4,2,health,services,...,yes,yes,yes,3,2,2,1,1,5,0
4,GP,F,16,U,GT3,T,3,3,other,other,...,yes,no,no,4,3,2,1,2,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644,MS,F,19,R,GT3,T,2,3,services,other,...,yes,yes,no,5,4,2,1,2,5,4
645,MS,F,18,U,LE3,T,3,1,teacher,services,...,yes,yes,no,4,3,4,1,1,1,4
646,MS,F,18,U,GT3,T,1,1,other,other,...,yes,no,no,1,1,1,1,1,5,6
647,MS,M,17,U,LE3,T,3,1,services,services,...,yes,yes,no,2,4,5,3,4,2,6


In [54]:
y

,G1,G2,G3
0,0,11,11
1,9,11,11
2,12,13,12
3,14,14,14
4,11,13,13
...,...,...,...
644,10,11,10
645,15,15,16
646,11,12,9
647,10,10,10


In [ ]:
y1 = y.iloc[:,0]
y2 = y.iloc[:,1]
y3 = y.iloc[:,2]
# separo as 3 colunas target

In [56]:
x.dtypes

school        object
sex           object
age            int64
address       object
famsize       object
Pstatus       object
Medu           int64
Fedu           int64
Mjob          object
Fjob          object
reason        object
guardian      object
traveltime     int64
studytime      int64
failures       int64
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic      object
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences       int64
dtype: object

In [ ]:
x = x.drop('address', axis = 1)
# retiro o endereço, por mais que possa ajudar na previsao

In [ ]:
sum =x.select_dtypes(include='object').columns
#soma da quantidade de colunas do tipo objeto

In [ ]:
Oh = ColumnTransformer(transformers=[('OneHot',OneHotEncoder(),sum)], remainder='passthrough')
# crio o one hot encoder para fazer a transformação das colunas categoricas
x = Oh.fit_transform(x)

In [ ]:
x.shape
# dimensao

(649, 54)

In [61]:
(54+3) /2

28.5

In [ ]:
cInput = Input(shape=(54,))
oculta1 = Dense(units=28, activation='relu')(cInput)
oculta2 = Dense(units=28, activation='relu')(oculta1)
saida1= Dense(units=1,activation='linear')(oculta2)
saida2= Dense(units=1,activation='linear')(oculta2)
saida3= Dense(units=1,activation='linear')(oculta2)
# mesmo estilo de 1 saida, porem agora com 3 saidas
# cada uma se referencia a outra, pega a camada anterior, como os outputs sao os ultimo a camada oculta2 é de onde vai saiir as 3 previsoes


In [ ]:
regressao = Model(inputs= cInput,outputs=[saida1,saida2,saida3])
# atribui ao mudelo os inputs e as 3 saidas
regressao.compile(optimizer='adam', loss = 'mse')

In [ ]:
regressao.fit(x, [y1, y2, y3], epochs=500, batch_size=100)
# treina o modelo 

Epoch 1/500
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - dense_12_loss: 57.9785 - dense_13_loss: 149.2932 - dense_14_loss: 120.5199 - loss: 329.4557
Epoch 2/500
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - dense_12_loss: 38.3730 - dense_13_loss: 125.0407 - dense_14_loss: 89.9179 - loss: 256.0551 
Epoch 3/500
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - dense_12_loss: 22.7179 - dense_13_loss: 101.7585 - dense_14_loss: 60.3113 - loss: 186.7466
Epoch 4/500
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - dense_12_loss: 13.1096 - dense_13_loss: 76.1347 - dense_14_loss: 34.6199 - loss: 125.8852
Epoch 5/500
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - dense_12_loss: 10.7788 - dense_13_loss: 50.9765 - dense_14_loss: 18.3753 - loss: 81.4737
Epoch 6/500
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - dense_12_loss: 12.1169 - dense_13_loss: 29.0844 - dense_14_loss: 14.6227 - loss: 55.9858
Epoch 7/500
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - dense_12_loss: 11.7506 - dense_13_loss: 15.3019 - dense_14_loss: 15.6227 - loss: 42.9604
Epoc

In [ ]:
predict1,predict2,predict3 = regressao.predict(x)
# faz a predição para as tres saidas

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [73]:
y1.mean(),y2.mean(), y3.mean()
# media das 3 notas 

(np.float64(11.399075500770415),
 np.float64(11.570107858243452),
 np.float64(11.906009244992296))

In [ ]:
predict1.mean(),predict2.mean(), predict3.mean()
# media das predições

(np.float32(11.119502), np.float32(11.2808075), np.float32(11.581188))

In [74]:
mean_absolute_error(y1,predict1),mean_absolute_error(y2,predict2),mean_absolute_error(y3,predict2)
#calcula os 3 MAE 

(np.float64(1.3714895388011021),
 np.float64(1.4268994647292033),
 np.float64(1.612679437422789))